In [17]:
import pandas as pd


In [18]:
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

In [19]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [20]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [21]:
def tokenize(text:str):
  text=  text.lower()
  text= text.replace(',','')
  text= text.replace('?','')
  text= text.replace('!','')
  return text.split()

In [9]:
# df.apply(lambda x: tokenize(x['question']),axis=1)

,0
0,"[what, is, the, capital, of, france]"
1,"[what, is, the, capital, of, germany]"
2,"[who, wrote, 'to, kill, a, mockingbird']"
3,"[what, is, the, largest, planet, in, our, sola..."
4,"[what, is, the, boiling, point, of, water, in,..."
...,...
85,"[who, directed, the, movie, 'titanic']"
86,"[which, superhero, is, also, known, as, the, d..."
87,"[what, is, the, capital, of, brazil]"
88,"[which, fruit, is, known, as, the, king, of, f..."


In [22]:
vocab = {
    '<UNK>':0
}

In [23]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab)

In [24]:
df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [25]:
len(vocab)

326

In [26]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 "'to": 12,
 'kill': 13,
 'a': 14,
 "mockingbird'": 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 "'1984'": 67,
 'george-orwell': 68,
 'currency': 69,
 '

In [28]:
def text_to_indices(text,vocab):
  indexed_text = []
  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])
  return indexed_text


In [29]:
text_to_indices('what is your name',vocab)

[1, 2, 0, 0]

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader

In [31]:
class QADataset(Dataset):

  def __init__(self, df, vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [32]:
dataset = QADataset(df, vocab)

In [33]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [34]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[  1,   2,   3,  37,  38,  39, 162]]) tensor([163])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[ 42, 252, 253, 118, 254, 255]]) tensor([256])
tensor([[ 42,   2,   3, 276, 212, 277]]) tensor([278])
tensor([[ 42, 137,   2, 227, 143,   3, 228, 229]]) tensor([156])
tensor([[ 10, 140,   3, 141, 142, 143, 144,  83,   3, 145]]) tensor([146])
tensor([[10, 75, 76]]) tensor([77])
tensor([[42, 43, 44, 45, 46, 47, 48]]) tensor([49])
tensor([[ 42, 137,   2,  62,  39,   3, 324, 325]]) tensor([6])
tensor([[  1,   2,   3, 164, 165, 166,  83,  84]]) tensor([167])
tensor([[  1,  87, 230, 231, 232, 233]]) tensor([234])
tensor([[1, 2, 3, 4, 5, 6]]) tensor([7])
tensor([[ 42, 168,   2,   3,  17, 169, 170]]) tensor([171])
tensor([[ 42,  18,   2,   3, 283, 143,   3, 284]]) tensor([206])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([36])
tensor([[ 78,  79, 196,  81,  19,   3, 197, 198, 199]]) tensor([200])
tensor([[  1,   2,   3, 213,   5,  14, 214, 215]]) tensor([216])
tensor([[ 

In [35]:
import torch.nn as nn

In [37]:
class Network(nn.Module):

  def __init__(self,vocab_size ):
    super().__init__()

    self.embedding=nn.Embedding(vocab_size,embedding_dim=60)
    self.rnn = nn.RNN(60,128,batch_first=True)
    self.fc = nn.Linear(128,vocab_size)

  def forward(self,question):
    embedded_question = self.embedding(question)
    hidden,final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [49]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [42]:
learning_rate = 0.001
epochs = 200

In [43]:
model = Network(len(vocab))

In [44]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [45]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 522.126061
Epoch: 2, Loss: 401.672246
Epoch: 3, Loss: 285.143569
Epoch: 4, Loss: 194.144117
Epoch: 5, Loss: 119.012067
Epoch: 6, Loss: 67.887553
Epoch: 7, Loss: 41.625174
Epoch: 8, Loss: 26.556608
Epoch: 9, Loss: 18.386896
Epoch: 10, Loss: 13.222540
Epoch: 11, Loss: 10.039887
Epoch: 12, Loss: 7.701115
Epoch: 13, Loss: 6.213035
Epoch: 14, Loss: 5.204472
Epoch: 15, Loss: 4.333184
Epoch: 16, Loss: 3.711536
Epoch: 17, Loss: 3.210122
Epoch: 18, Loss: 2.817637
Epoch: 19, Loss: 2.483774
Epoch: 20, Loss: 2.220196
Epoch: 21, Loss: 1.990471
Epoch: 22, Loss: 1.785305
Epoch: 23, Loss: 1.613357
Epoch: 24, Loss: 1.460696
Epoch: 25, Loss: 1.332195
Epoch: 26, Loss: 1.218125
Epoch: 27, Loss: 1.116750
Epoch: 28, Loss: 1.028226
Epoch: 29, Loss: 0.947788
Epoch: 30, Loss: 0.877153
Epoch: 31, Loss: 0.809152
Epoch: 32, Loss: 0.751882
Epoch: 33, Loss: 0.698010
Epoch: 34, Loss: 0.651712
Epoch: 35, Loss: 0.605587
Epoch: 36, Loss: 0.564105
Epoch: 37, Loss: 0.528008
Epoch: 38, Loss: 0.494319
Epoch

In [46]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [47]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [48]:
list(vocab.keys())[7]

'paris'